[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C32_Skills_Tools_Course/03_mcp_server/03_mcp_server.ipynb)

# 03 · 从零造 MCP Server（不依赖框架）

目标：用**纯标准库（只用 json）**从零写一个 **mini MCP**——JSON-RPC 2.0 消息编解码 → mini server（tools/resources）→ mini client（握手+列举+调用）→ **端到端闭环 + 对拍 + stdio 传输模拟**，全程 `assert` 验证、**无需 API key、不联网、不依赖任何 MCP SDK**。

路线：JSON-RPC 编解码 → mini server → mini client + 握手 → 端到端对拍 → stdio 传输模拟 → 错误协商 → ✏️ 练习 → 📖 答案 → 🧪 真实 MCP 报文胶囊 → 真实 Claude 旁注。

> 心智模型：**MCP = 给「AI 应用 ↔ 外部工具」立一个开放标准**。它不改变工具调用的本质，只标准化「工具怎么被发现、被描述、消息怎么传」。我们亲手造一个，就彻底懂了。

## 1 · JSON-RPC 2.0 消息编解码

MCP 的消息是 JSON-RPC 2.0，只有三种：**请求**（带 id+method+params）、**响应**（带 id + result 或 error）、**通知**（无 id）。

先把它们的构造函数写出来——这是后面一切的字典级地基。标准错误码：`-32700` 解析、`-32600` 非法请求、`-32601` 方法不存在、`-32602` 参数非法、`-32603` 内部错误。

In [ ]:
import json

JSONRPC = '2.0'
# 标准错误码（JSON-RPC 2.0 规范）
PARSE_ERROR, INVALID_REQUEST = -32700, -32600
METHOD_NOT_FOUND, INVALID_PARAMS, INTERNAL_ERROR = -32601, -32602, -32603

def make_request(method, params=None, id=1):
    '''请求：要对方执行 method 并回结果。带 id 用于配对。'''
    return {'jsonrpc': JSONRPC, 'id': id, 'method': method, 'params': params or {}}

def make_response(id, result):
    '''成功响应：用相同的 id 指回，携带 result。'''
    return {'jsonrpc': JSONRPC, 'id': id, 'result': result}

def make_error(id, code, message, data=None):
    '''失败响应：用相同的 id 指回，携带 error{code,message}。'''
    err = {'code': code, 'message': message}
    if data is not None:
        err['data'] = data
    return {'jsonrpc': JSONRPC, 'id': id, 'error': err}

def make_notification(method, params=None):
    '''通知：发了不等回复，所以没有 id。'''
    return {'jsonrpc': JSONRPC, 'method': method, 'params': params or {}}

req = make_request('tools/list', {}, id=1)
ok = make_response(1, {'tools': []})
bad = make_error(1, METHOD_NOT_FOUND, 'Method not found')
note = make_notification('notifications/initialized')
print('请求 :', req)
print('成功 :', ok)
print('失败 :', bad)
print('通知 :', note)
# 不变量：版本恒为 2.0；请求/响应有 id；通知无 id；响应 result/error 二选一
assert all(m['jsonrpc'] == '2.0' for m in (req, ok, bad, note))
assert req['id'] == 1 and ok['id'] == 1 and bad['id'] == 1
assert 'id' not in note, '通知不能有 id'
assert ('result' in ok) != ('result' in bad)   # 成功有 result，失败没有
assert 'error' in bad and bad['error']['code'] == -32601
print('✅ JSON-RPC 三种消息构造正确：请求/响应(result|error)/通知(无id)')

## 2 · 解析与分类：从一行 JSON 文本到消息类型

真实里消息是**一行 JSON 文本**（stdio 传输）。我们要能把一行文本解析成字典，并判断它是**请求/响应/通知**中的哪一种——这是 server 收到字节后第一件要做的事。

解析失败（坏 JSON）要能优雅识别（对应 `-32700`），而不是让程序崩溃。

In [ ]:
def parse_message(line):
    '''把一行文本解析成 (kind, obj)。kind in {request,response,notification,parse_error}。'''
    try:
        obj = json.loads(line)
    except json.JSONDecodeError:
        return ('parse_error', None)        # 对应 -32700
    if not isinstance(obj, dict) or obj.get('jsonrpc') != '2.0':
        return ('invalid', obj)             # 对应 -32600
    has_id = 'id' in obj
    has_method = 'method' in obj
    if has_method and has_id:
        return ('request', obj)
    if has_method and not has_id:
        return ('notification', obj)
    if not has_method and has_id:
        return ('response', obj)            # 有 id 无 method => 是响应
    return ('invalid', obj)

print(parse_message('{"jsonrpc":"2.0","id":1,"method":"tools/list"}')[0])
print(parse_message('{"jsonrpc":"2.0","method":"notifications/initialized"}')[0])
print(parse_message('{"jsonrpc":"2.0","id":1,"result":{}}')[0])
print(parse_message('not json at all')[0])
assert parse_message('{"jsonrpc":"2.0","id":1,"method":"x"}')[0] == 'request'
assert parse_message('{"jsonrpc":"2.0","method":"x"}')[0] == 'notification'
assert parse_message('{"jsonrpc":"2.0","id":1,"result":{}}')[0] == 'response'
assert parse_message('{bad json')[0] == 'parse_error'
assert parse_message('{"id":1}')[0] == 'invalid'    # 缺 jsonrpc 版本
print('✅ 解析器就绪：请求/响应/通知正确分类，坏 JSON 不崩溃(parse_error)')

## 3 · mini MCP server：tools + resources

server 持有一批 **tools**（name→{def, fn}）和 **resources**（uri→content），用一个 `handle(request)→response` 方法按 `method` 分发：
`initialize` / `tools/list` / `tools/call` / `resources/list` / `resources/read`。

注意 `tools/call` 的内核**就是一个工具分发器**——MCP 只是套了层 JSON-RPC 信封。

In [ ]:
PROTOCOL_VERSION = '2024-11-05'   # MCP 协议版本（贴近真实日期版本号）

class MiniMCPServer:
    def __init__(self, name='mini-server'):
        self.name = name
        self.tools = {}        # name -> {'def':..., 'fn':...}
        self.resources = {}    # uri  -> {'name':..., 'text':...}
    def add_tool(self, definition, fn):
        self.tools[definition['name']] = {'def': definition, 'fn': fn}
    def add_resource(self, uri, name, text):
        self.resources[uri] = {'name': name, 'text': text}
    # —— 核心：把一条 JSON-RPC 请求处理成一条响应 ——
    def handle(self, request):
        rid, method, params = request.get('id'), request.get('method'), request.get('params', {})
        try:
            if method == 'initialize':
                return make_response(rid, {
                    'protocolVersion': PROTOCOL_VERSION,
                    'serverInfo': {'name': self.name},
                    'capabilities': {'tools': {}, 'resources': {}}})   # 声明我支持 tools+resources
            if method == 'tools/list':
                return make_response(rid, {'tools': [t['def'] for t in self.tools.values()]})
            if method == 'tools/call':
                return make_response(rid, self._call_tool(params))
            if method == 'resources/list':
                return make_response(rid, {'resources': [
                    {'uri': u, 'name': r['name']} for u, r in self.resources.items()]})
            if method == 'resources/read':
                return make_response(rid, self._read_resource(params))
            return make_error(rid, METHOD_NOT_FOUND, f'Method not found: {method}')
        except KeyError as e:
            return make_error(rid, INVALID_PARAMS, str(e))
    def _call_tool(self, params):
        name = params['name']                         # 缺 name -> KeyError -> -32602
        if name not in self.tools:
            raise KeyError(f'unknown tool: {name}')
        args = params.get('arguments', {})
        try:
            out = self.tools[name]['fn'](**args)       # 真正执行那个函数
            return {'content': [{'type': 'text', 'text': str(out)}], 'isError': False}
        except Exception as e:                        # 执行错误作为内容回传
            return {'content': [{'type': 'text', 'text': f'{type(e).__name__}: {e}'}], 'isError': True}
    def _read_resource(self, params):
        uri = params['uri']
        if uri not in self.resources:
            raise KeyError(f'unknown resource: {uri}')
        return {'contents': [{'uri': uri, 'text': self.resources[uri]['text']}]}

# 建一个 server，加一个 add 工具和一个文件资源
def add(a, b):
    return a + b
ADD_DEF = {'name': 'add', 'description': '两数相加',
           'inputSchema': {'type': 'object',
                           'properties': {'a': {'type': 'number'}, 'b': {'type': 'number'}},
                           'required': ['a', 'b']}}
srv = MiniMCPServer()
srv.add_tool(ADD_DEF, add)
srv.add_resource('file:///readme.md', 'readme', '# Hello MCP')

resp_init = srv.handle(make_request('initialize', {'protocolVersion': PROTOCOL_VERSION}, 1))
resp_list = srv.handle(make_request('tools/list', {}, 2))
print('initialize ->', resp_init['result']['capabilities'])
print('tools/list ->', [t['name'] for t in resp_list['result']['tools']])
assert resp_init['result']['protocolVersion'] == PROTOCOL_VERSION
assert 'tools' in resp_init['result']['capabilities']
assert resp_list['result']['tools'][0]['name'] == 'add'
print('✅ mini server 就绪：能 initialize、能 tools/list')

## 4 · mini MCP client：强制先握手

client 包住一个 server（真实里是包住一个传输连接），提供 `initialize()` / `list_tools()` / `call_tool()` / `list_resources()` / `read_resource()`。

**纪律**：握手前调用任何业务方法都应报错——这是有状态协议的通用规矩（先握手、再办事）。

In [ ]:
class MiniMCPClient:
    def __init__(self, server):
        self.server = server          # 真实里这里是一个 transport
        self.initialized = False
        self.server_capabilities = None
        self._id = 0
    def _next_id(self):
        self._id += 1
        return self._id
    def _rpc(self, method, params=None):
        '''发一条请求、拿一条响应；error 则抛出。'''
        resp = self.server.handle(make_request(method, params or {}, self._next_id()))
        if 'error' in resp:
            raise RuntimeError(f"RPC error {resp['error']['code']}: {resp['error']['message']}")
        return resp['result']
    def initialize(self):
        '''三拍握手：initialize 请求 -> 存能力 -> initialized 通知。'''
        result = self._rpc('initialize', {'protocolVersion': PROTOCOL_VERSION,
                                          'capabilities': {}})
        self.server_capabilities = result['capabilities']
        self.server.handle(make_notification('notifications/initialized'))  # 通知，不等回复
        self.initialized = True
        return result
    def _require_init(self):
        if not self.initialized:
            raise RuntimeError('必须先 initialize() 再调用业务方法')
    def list_tools(self):
        self._require_init()
        return self._rpc('tools/list')['tools']
    def call_tool(self, name, arguments):
        self._require_init()
        return self._rpc('tools/call', {'name': name, 'arguments': arguments})
    def list_resources(self):
        self._require_init()
        return self._rpc('resources/list')['resources']
    def read_resource(self, uri):
        self._require_init()
        return self._rpc('resources/read', {'uri': uri})

cli = MiniMCPClient(srv)
# 握手前调用应报错
blocked = False
try:
    cli.list_tools()
except RuntimeError:
    blocked = True
assert blocked, '握手前调用业务方法必须报错'
info = cli.initialize()
print('握手成功, server 能力:', cli.server_capabilities)
assert cli.initialized is True
assert 'tools' in cli.server_capabilities
print('✅ mini client 就绪：强制先握手；握手后存下 server 能力')

## 5 · 端到端闭环 + 对拍：MCP 调用 == 直接调用

完整跑一遍 **发现 → 调用**：`initialize → list_tools → call_tool('add')`，并**对拍**——经 MCP 调 `add(3,4)` 的结果，应与**直接调用** `add(3,4)` 完全一致。

这就是本课的「对拍」：MCP 是协议层的包装，不改变计算本质。逻辑对了，就能原样搬到真实 MCP。

In [ ]:
# 1) 发现
tools = cli.list_tools()
print('发现的工具:', [t['name'] for t in tools])
assert any(t['name'] == 'add' for t in tools)

# 2) 调用
res = cli.call_tool('add', {'a': 3, 'b': 4})
print('tools/call 结果:', res)
mcp_value = res['content'][0]['text']        # MCP 把结果装在 content[].text

# 3) 对拍：MCP 调用 == 直接调用
direct_value = str(add(3, 4))
print(f'MCP 路径={mcp_value!r}  直接调用={direct_value!r}')
assert mcp_value == direct_value == '7'
assert res['isError'] is False

# 4) 资源读取
rd = cli.read_resource('file:///readme.md')
assert rd['contents'][0]['text'] == '# Hello MCP'
print('resources/read ->', rd['contents'][0]['text'])
print('✅ 端到端闭环跑通；MCP 调用结果与直接调用逐字一致（对拍成功）')

## 6 · stdio 传输模拟：换行分隔的 JSON

本地 MCP server 常用 **stdio**：消息按行（换行符分隔）写进对方 stdin、从 stdout 读回。
我们用**两个内存队列**当一对管道，证明「client 写一行请求 → server 读到、处理 → 写一行响应 → client 读回」的闭环——**协议内容不变，只是换了传输**。

In [ ]:
from collections import deque

class StdioPipe:
    '''内存模拟一对 stdio 管道：每条消息序列化成一行 JSON。'''
    def __init__(self):
        self.c2s = deque()   # client -> server（server 的 stdin）
        self.s2c = deque()   # server -> client（server 的 stdout）
    def write_line(self, q, obj):
        q.append(json.dumps(obj, ensure_ascii=False) + '\n')   # newline-delimited
    def read_line(self, q):
        line = q.popleft()
        assert line.endswith('\n'), '每条消息以换行结尾'
        return json.loads(line)

def serve_over_stdio(server, pipe):
    '''server 侧循环：从 stdin 读一行请求、handle、把响应写到 stdout。处理完所有待处理请求。'''
    while pipe.c2s:
        request = pipe.read_line(pipe.c2s)
        if 'id' not in request:        # 通知：处理但不回复
            continue
        response = server.handle(request)
        pipe.write_line(pipe.s2c, response)

pipe = StdioPipe()
srv2 = MiniMCPServer(); srv2.add_tool(ADD_DEF, add)

# client 写两条请求到 stdin（序列化成行）
pipe.write_line(pipe.c2s, make_request('initialize', {'protocolVersion': PROTOCOL_VERSION}, 1))
pipe.write_line(pipe.c2s, make_request('tools/call', {'name': 'add', 'arguments': {'a': 10, 'b': 5}}, 2))
# server 处理这两行
serve_over_stdio(srv2, pipe)
# client 从 stdout 读回响应
r1 = pipe.read_line(pipe.s2c)
r2 = pipe.read_line(pipe.s2c)
print('读回响应1(initialize):', r1['result']['protocolVersion'])
print('读回响应2(tools/call):', r2['result']['content'][0]['text'])
assert r1['id'] == 1 and r2['id'] == 2          # id 配对，顺序对齐
assert r2['result']['content'][0]['text'] == '15'
print('✅ stdio 闭环跑通：写行->对面读->处理->写行->读回；协议不变，仅换传输')

## 7 · 错误与能力协商

健壮的 server 要对**未知方法**回 `-32601 Method not found`、对**缺参/未知工具**回 `-32602 Invalid params`；client 要能把这些 error 浮现给上层。

关键区分：**协议层错误**（未知方法/缺参）走 JSON-RPC `error`；**工具执行错误**走 result 里的 `isError`（让模型看得到、能改）。

In [ ]:
# 未知方法 -> -32601
resp_unknown = srv.handle(make_request('does/not/exist', {}, 9))
print('未知方法 ->', resp_unknown['error'])
assert resp_unknown['error']['code'] == METHOD_NOT_FOUND

# tools/call 缺 name -> -32602
resp_noname = srv.handle(make_request('tools/call', {}, 10))
assert resp_noname['error']['code'] == INVALID_PARAMS

# 未知工具 -> 也归为 -32602（参数 name 非法）
resp_badtool = srv.handle(make_request('tools/call', {'name': 'nope', 'arguments': {}}, 11))
assert resp_badtool['error']['code'] == INVALID_PARAMS

# 工具执行抛错 -> 不是 RPC error，而是 result 里 isError=True（让模型看到错误内容）
def boom(x):
    raise ValueError('炸了')
srv.add_tool({'name': 'boom', 'description': 'always fails',
              'inputSchema': {'type': 'object', 'properties': {'x': {'type': 'number'}}, 'required': ['x']}}, boom)
resp_boom = srv.handle(make_request('tools/call', {'name': 'boom', 'arguments': {'x': 1}}, 12))
assert 'error' not in resp_boom                    # 执行错误不是协议层 error
assert resp_boom['result']['isError'] is True      # 而是结果里的 isError
assert 'ValueError' in resp_boom['result']['content'][0]['text']

# client 把协议层 error 抛成异常
raised = False
try:
    cli._rpc('does/not/exist')
except RuntimeError as e:
    raised = '32601' in str(e)
assert raised
print('✅ 错误协商正确：未知方法/缺参 -> RPC error；工具执行失败 -> result.isError')

---
## ✏️ 练习 1：构造并校验一条 JSON-RPC 请求

MCP 的字典级地基是「构造合法消息」。实现 `build_call_request(tool_name, arguments, id)`：返回一条**合法的 `tools/call` 请求**字典，字段需满足 JSON-RPC 2.0：`jsonrpc=='2.0'`、有 `id`、`method=='tools/call'`、`params=={'name':..., 'arguments':...}`。

In [ ]:
def build_call_request(tool_name, arguments, id):
    # TODO: 返回 {'jsonrpc':'2.0', 'id':id, 'method':'tools/call',
    #             'params':{'name':tool_name, 'arguments':arguments}}
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
r = build_call_request('add', {'a': 1, 'b': 2}, id=7)
assert r['jsonrpc'] == '2.0' and r['id'] == 7
assert r['method'] == 'tools/call'
assert r['params'] == {'name': 'add', 'arguments': {'a': 1, 'b': 2}}
# 而且我们的 server 能直接处理它
resp = srv.handle(r)
assert resp['result']['content'][0]['text'] == '3'
print('✅ 练习 1 通过：构造的请求合法, 且能被 server 正确执行')

## ✏️ 练习 2：客户端握手校验

健壮的 client 在握手后应**校验** server 回的东西：协议版本要匹配、capabilities 要存在。

实现 `verify_handshake(init_result, expected_version)`：检查 `protocolVersion == expected_version` 且 `capabilities` 是个非空 dict；通过返回 `True`，否则抛 `ValueError`（信息含原因）。

In [ ]:
def verify_handshake(init_result, expected_version):
    # TODO:
    #   - init_result['protocolVersion'] != expected_version -> raise ValueError('协议版本不匹配...')
    #   - 'capabilities' 不在 init_result 或不是 dict 或为空 -> raise ValueError('缺少 capabilities')
    #   - 否则 return True
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
good_init = {'protocolVersion': PROTOCOL_VERSION, 'capabilities': {'tools': {}}}
assert verify_handshake(good_init, PROTOCOL_VERSION) is True
# 版本不匹配
bad_ver = False
try:
    verify_handshake({'protocolVersion': '1999-01-01', 'capabilities': {}}, PROTOCOL_VERSION)
except ValueError:
    bad_ver = True
assert bad_ver
# 缺 capabilities
no_cap = False
try:
    verify_handshake({'protocolVersion': PROTOCOL_VERSION}, PROTOCOL_VERSION)
except ValueError:
    no_cap = True
assert no_cap
print('✅ 练习 2 通过：版本匹配 + capabilities 存在才放行')

## ✏️ 练习 3：在 tools/call 里加 schema 校验

现在的 server 直接把 `arguments` 喂给函数，没校验。请写一个**带 schema 校验**的 tools/call 处理：调用前用 `required` 检查必填字段，缺字段时返回 `result` 里 `isError=True`、内容含缺失字段名（**不抛 RPC error**，让模型能看到并改）。

实现 `call_tool_validated(server, name, arguments)`：返回 tools/call 的 `result` 部分。

In [ ]:
def call_tool_validated(server, name, arguments):
    # TODO:
    #   - name 不在 server.tools -> 返回 {'content':[{'type':'text','text':f'unknown tool: {name}'}], 'isError':True}
    #   - 取该工具 inputSchema 的 required，逐个检查 arguments 是否都有；
    #     缺字段 -> 返回 isError=True，内容含 f'缺少必填字段: {k}'
    #   - 校验通过 -> 执行 fn(**arguments)，正常返回 {'content':[...], 'isError':False}
    #     执行抛错 -> isError=True，内容为错误信息
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
ok_res = call_tool_validated(srv, 'add', {'a': 1, 'b': 2})
assert ok_res['isError'] is False and ok_res['content'][0]['text'] == '3'
# 缺必填 b -> isError + 提示
miss_res = call_tool_validated(srv, 'add', {'a': 1})
assert miss_res['isError'] is True and 'b' in miss_res['content'][0]['text']
# 未知工具 -> isError
unk_res = call_tool_validated(srv, 'ghost', {})
assert unk_res['isError'] is True and 'unknown' in unk_res['content'][0]['text'].lower()
print('✅ 练习 3 通过：tools/call 前置 schema 校验，缺参作为 isError 内容回传')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def build_call_request(tool_name, arguments, id):
    return {'jsonrpc': '2.0', 'id': id, 'method': 'tools/call',
            'params': {'name': tool_name, 'arguments': arguments}}

In [ ]:
# 练习 2 参考答案
def verify_handshake(init_result, expected_version):
    if init_result.get('protocolVersion') != expected_version:
        raise ValueError(f"协议版本不匹配: 期望 {expected_version}, 得到 {init_result.get('protocolVersion')}")
    caps = init_result.get('capabilities')
    if not isinstance(caps, dict) or len(caps) == 0:
        raise ValueError('缺少 capabilities')
    return True

In [ ]:
# 练习 3 参考答案
def call_tool_validated(server, name, arguments):
    if name not in server.tools:
        return {'content': [{'type': 'text', 'text': f'unknown tool: {name}'}], 'isError': True}
    schema = server.tools[name]['def'].get('inputSchema', {})
    for k in schema.get('required', []):
        if k not in arguments:
            return {'content': [{'type': 'text', 'text': f'缺少必填字段: {k}'}], 'isError': True}
    try:
        out = server.tools[name]['fn'](**arguments)
        return {'content': [{'type': 'text', 'text': str(out)}], 'isError': False}
    except Exception as e:
        return {'content': [{'type': 'text', 'text': f'{type(e).__name__}: {e}'}], 'isError': True}

---
## 🧪 真实数据胶囊：真实 MCP 报文的形状

下面是**贴近 MCP 规范**的真实报文：一次 `initialize` 请求/响应、一次 `tools/call` 请求/响应。我们用上面**从零写的 server 与解析器**去处理它们——字段几乎一一对应。这就是「你自己造的 scaffold 可直接迁移到真实 MCP」的证据。

> 来源形状：modelcontextprotocol.io 规范中的 JSON-RPC 报文样例（字段：jsonrpc / id / method / params / result）。

In [ ]:
# 真实 MCP 报文样例（捕获自一次会话，形如规范示例）
REAL_LOG = [
    {'jsonrpc': '2.0', 'id': 1, 'method': 'initialize',
     'params': {'protocolVersion': '2024-11-05',
                'capabilities': {'roots': {'listChanged': True}},
                'clientInfo': {'name': 'ExampleClient', 'version': '1.0.0'}}},
    {'jsonrpc': '2.0', 'id': 1,
     'result': {'protocolVersion': '2024-11-05',
                'capabilities': {'tools': {'listChanged': True}, 'resources': {}},
                'serverInfo': {'name': 'ExampleServer', 'version': '1.0.0'}}},
    {'jsonrpc': '2.0', 'method': 'notifications/initialized'},
    {'jsonrpc': '2.0', 'id': 2, 'method': 'tools/list', 'params': {}},
    {'jsonrpc': '2.0', 'id': 3, 'method': 'tools/call',
     'params': {'name': 'get_weather', 'arguments': {'location': 'Beijing'}}},
    {'jsonrpc': '2.0', 'id': 3,
     'result': {'content': [{'type': 'text', 'text': 'Sunny, 26C'}], 'isError': False}},
]

# 用我们写的 parse_message 给每条报文分类——证明同形
kinds = [parse_message(json.dumps(m))[0] for m in REAL_LOG]
print('真实报文逐条类型:', kinds)
assert kinds == ['request', 'response', 'notification', 'request', 'request', 'response']
init_resp = REAL_LOG[1]
assert 'tools' in init_resp['result']['capabilities']            # server 声明了 tools 能力
call_resp = REAL_LOG[5]
assert call_resp['result']['content'][0]['text'] == 'Sunny, 26C'  # 结果在 content[].text
print('真实 initialize 协议版本:', init_resp['result']['protocolVersion'])
print('真实 tools/call 结果   :', call_resp['result']['content'][0]['text'])
print('✅ 真实 MCP 报文与本课从零实现同形（字段一一对应，解析器直接吃下）')

**🧪 胶囊练习**：实现 `method_names(log)`：从一段报文日志里提取**所有出现过的 method 名**（请求与通知都有 method；响应没有），去重后按首次出现顺序返回列表。（分析真实 MCP 会话「用了哪些方法」就是这么做的。）

In [ ]:
def method_names(log):
    # TODO: 遍历 log，收集每条消息的 'method'(若有)，去重保序后返回 list
    raise NotImplementedError

In [ ]:
# 自测
names = method_names(REAL_LOG)
assert names == ['initialize', 'notifications/initialized', 'tools/list', 'tools/call']
print('会话中用到的方法:', names)
print('✅ 胶囊练习通过')

In [ ]:
# 📖 胶囊参考答案
def method_names(log):
    seen = []
    for m in log:
        meth = m.get('method')
        if meth is not None and meth not in seen:
            seen.append(meth)
    return seen

---
## 🔧 旁注：把你的 mini server 接到真实 Claude / MCP

本课用内存模拟跑通的「握手 → 列举 → 调用」，在真实世界有两个落点（伪代码，**本环境不跑**；无 key 时本课所有真实适配都自动回退 MockLLM，绝不阻断）：

**(1) 真实 MCP 客户端**（Python `mcp` 包），把传输换成真 stdio：
```python
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
params = StdioServerParameters(command='your-mcp-server', args=[])
async with stdio_client(params) as (read, write):          # 真 stdio 管道
    async with ClientSession(read, write) as session:
        await session.initialize()                         # == 我们的三拍握手
        tools = await session.list_tools()                 # == 我们的 tools/list
        result = await session.call_tool('add', {'a': 3, 'b': 4})  # == 我们的 tools/call
```

**(2) 真实 Claude 直连远程 MCP server**（Messages API 的 MCP connector，beta header `mcp-client-2025-11-20`）——
你写的 mini server 部署成一个真实 server 后，可以被真实 host 这样接入：
```python
import anthropic
client = anthropic.Anthropic()                              # 读 ANTHROPIC_API_KEY
resp = client.beta.messages.create(
    model='claude-sonnet-4-6', max_tokens=1024,
    betas=['mcp-client-2025-11-20'],
    mcp_servers=[{'type': 'url', 'name': 'mine', 'url': 'https://my-server/mcp'}],
    tools=[{'type': 'mcp_toolset', 'mcp_server_name': 'mine'}],   # 引用上面的 server
    messages=[{'role': 'user', 'content': '帮我算 3+4'}],
)
```

对应关系：`session.initialize()` ↔ 三拍握手、`list_tools`/`call_tool` ↔ 我们的同名方法、返回的 `content[].text` 形状**完全一致**。你在内存里验证过的 server 处理逻辑（`handle`）原样可用——把传输换成真管道、把 MockLLM 换成 `messages.create(model=...)` 即可。

### 小结
- MCP = 给「AI 应用 ↔ 外部工具/数据」立开放标准：把 M×N 私有集成坍缩成 M+N；我们**只用标准库 json 就从零造了一个**。
- 三角色：**host**（跑LLM）/ **client**（一对一连接器）/ **server**（暴露能力）；三能力：**tools**(动词)/**resources**(名词)/**prompts**(模板)。
- 消息走 **JSON-RPC 2.0**：请求(id+method+params)/响应(id+result|error)/通知(无id)；错误码 -32600~-32603/-32700。
- 流程：**先握手**(initialize+能力协商+initialized) → **发现**(tools/list) → **调用**(tools/call)。
- **tools/call 内核 = 工具分发器**：MCP 调用结果与直接调用逐字一致（对拍成功）。
- 传输层解耦：stdio(换行分隔JSON) / HTTP，协议内容不变；⚠️ server 返回值是注入攻击面，只接可信 server。

下一站：**模块 04 · 工具打包与插件** —— 把 skill、命令、工具、MCP 配置打包成可分发、可隔离的插件单元。